<img src="http://hilpisch.com/tpq_logo.png" alt="The Python Quants" width="35%" align="right" border="0"><br>

# Python for Algorithmic Trading 

**Chapter 05 &mdash; Predicting Market Movements with Machine Learning**

## Using Linear Regression for Market Movement Prediction

### A Quick Review of Linear Regression

In [ ]:
!git clone https://github.com/tpq-classes/python_for_algo_trading_core.git
import sys
sys.path.append('python_for_algo_trading_core')


In [ ]:
import os
import random
import numpy as np
from pylab import mpl, plt
plt.style.use('seaborn-v0_8')
mpl.rcParams['font.family'] = 'serif'
os.environ['PYTHONHASHSEED'] = '0'

In [ ]:
x = np.linspace(0, 10)
x

In [ ]:
def set_seeds(seed=100):
    random.seed(seed)
    np.random.seed(seed)
set_seeds()

In [ ]:
y = x + np.random.standard_normal(len(x))
y

In [ ]:
reg = np.polyfit(x, y, deg=1)

In [ ]:
reg

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(x, y, 'bo', label='data')
plt.plot(x, np.polyval(reg, x), 'r', lw=2.5,
         label='linear regression')
plt.legend(loc=0);

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(x, y, 'bo', label='data')
xn = np.linspace(0, 20)
plt.plot(xn, np.polyval(reg, xn), 'r', lw=2.5,
         label='linear regression')
plt.legend(loc=0);

### The Basic Idea for Price Prediction

In [ ]:
x = np.arange(12)

In [ ]:
x

In [ ]:
lags = 3

In [ ]:
m = np.zeros((lags + 1, len(x) - lags))

In [ ]:
m

In [ ]:
m[lags] = x[lags:]
for i in range(lags):
    m[i] = x[i:i - lags]

In [ ]:
m.T

In [ ]:
reg = np.linalg.lstsq(m[:lags].T, m[lags], rcond=None)[0]

In [ ]:
reg

In [ ]:
np.dot(m[:lags].T, reg)

### Predicting Index Levels

In [ ]:
import pandas as pd

In [ ]:
raw = pd.read_csv('http://hilpisch.com/pyalgo_eikon_eod_data.csv',
                  index_col=0, parse_dates=True).dropna()

In [ ]:
raw.info()

In [ ]:
symbol = 'EUR='

In [ ]:
data = pd.DataFrame(raw[symbol])

In [ ]:
data.rename(columns={symbol: 'price'}, inplace=True)

In [ ]:
lags = 5

In [ ]:
cols = []
for lag in range(1, lags + 1):
    col = f'lag_{lag}'
    data[col] = data['price'].shift(lag) # <1>
    cols.append(col)
data.dropna(inplace=True)

In [ ]:
data.head()

In [ ]:
reg = np.linalg.lstsq(data[cols], data['price'],
                      rcond=None)[0]

In [ ]:
reg

In [ ]:
data['prediction'] = np.dot(data[cols], reg)

In [ ]:
data[['price', 'prediction']].plot(figsize=(10, 6));

In [ ]:
data[['price', 'prediction']].loc['2019-10-1':].plot(
            figsize=(10, 6));

### Predicting Future Returns

In [ ]:
data['return'] = np.log(data['price'] /
                         data['price'].shift(1))

In [ ]:
data.dropna(inplace=True)

In [ ]:
cols = []
for lag in range(1, lags + 1):
    col = f'lag_{lag}'
    data[col] = data['return'].shift(lag)
    cols.append(col)
data.dropna(inplace=True)

In [ ]:
reg = np.linalg.lstsq(data[cols], data['return'],
                      rcond=None)[0]

In [ ]:
reg

In [ ]:
data['prediction'] = np.dot(data[cols], reg)

In [ ]:
data[['return', 'prediction']].iloc[lags:].plot(figsize=(10, 6));

In [ ]:
hits = np.sign(data['return'] *
               data['prediction']).value_counts()

In [ ]:
hits

In [ ]:
hits.values[0] / sum(hits)

### Prediction Future Market Direction

In [ ]:
reg = np.linalg.lstsq(data[cols], np.sign(data['return']),
                      rcond=None)[0]

In [ ]:
reg

In [ ]:
data['prediction'] = np.sign(np.dot(data[cols], reg))

In [ ]:
data['prediction'].head(5)

In [ ]:
data['prediction'].value_counts()

In [ ]:
hits = np.sign(data['return'] *
               data['prediction']).value_counts()

In [ ]:
hits

In [ ]:
hits.values[0] / sum(hits)

### Vectorized Backtesting of Regression-based Strategy 

In [ ]:
data.head()

In [ ]:
data['strategy'] = data['prediction'] * data['return']

In [ ]:
data[['return', 'strategy']].sum().apply(np.exp)

In [ ]:
data[['return', 'strategy']].dropna().cumsum(
        ).apply(np.exp).plot(figsize=(10, 6));

### Generalizing the Approach

In [ ]:
import LRVectorBacktester as LR

In [ ]:
lrbt = LR.LRVectorBacktester('EUR=', '2010-1-1', '2019-12-31',
                                     10000, 0.0)

In [ ]:
lrbt.run_strategy('2010-1-1', '2019-12-31',
                  '2010-1-1', '2019-12-31', lags=5)

In [ ]:
lrbt.run_strategy('2010-1-1', '2017-12-31',
                  '2018-1-1', '2019-12-31', lags=5)

In [ ]:
lrbt.plot_results()

In [ ]:
lrbt = LR.LRVectorBacktester('GDX', '2010-1-1', '2019-12-31',
                                     10000, 0.002)

In [ ]:
lrbt.run_strategy('2010-1-1', '2019-12-31',
                  '2010-1-1', '2019-12-31', lags=7)

In [ ]:
lrbt.run_strategy('2010-1-1', '2014-12-31',
                  '2015-1-1', '2019-12-31', lags=7)

In [ ]:
lrbt.plot_results()

## Using Machine Learning for Market Movement Prediction

### Linear Regression with scikit-learn

In [ ]:
x = np.arange(12)

In [ ]:
x

In [ ]:
lags = 3

In [ ]:
m = np.zeros((lags + 1, len(x) - lags))

In [ ]:
m[lags] = x[lags:]
for i in range(lags):
    m[i] = x[i:i - lags]

In [ ]:
from sklearn import linear_model

In [ ]:
lm = linear_model.LinearRegression()

In [ ]:
lm.fit(m[:lags].T, m[lags])

In [ ]:
lm.coef_

In [ ]:
lm.intercept_

In [ ]:
lm.predict(m[:lags].T)

In [ ]:
lm = linear_model.LinearRegression(fit_intercept=False)

In [ ]:
lm.fit(m[:lags].T, m[lags])

In [ ]:
lm.coef_

In [ ]:
lm.intercept_

In [ ]:
lm.predict(m[:lags].T)

### A Simple Classification Problem

In [ ]:
hours = np.array([0.5, 0.75, 1., 1.25, 1.5, 1.75, 1.75, 2.,
                  2.25, 2.5, 2.75, 3., 3.25, 3.5, 4., 4.25,
                  4.5, 4.75, 5., 5.5])

In [ ]:
success = np.array([0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 1, 0, 1,
                    0, 1, 1, 1, 1, 1, 1])

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(hours, success, 'ro')
plt.ylim(-0.2, 1.2);

In [ ]:
reg = np.polyfit(hours, success, deg=1)

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(hours, success, 'ro')
plt.plot(hours, np.polyval(reg, hours), 'b')
plt.ylim(-0.2, 1.2);

In [ ]:
lm = linear_model.LogisticRegression(solver='lbfgs')

In [ ]:
hrs = hours.reshape(1, -1).T

In [ ]:
lm.fit(hrs, success)

In [ ]:
prediction = lm.predict(hrs)

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(hours, success, 'ro', label='data')
plt.plot(hours, prediction, 'b', label='prediction')
plt.legend(loc=0)
plt.ylim(-0.2, 1.2);

In [ ]:
prob = lm.predict_proba(hrs)

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(hours, success, 'ro')
plt.plot(hours, prediction, 'b')
plt.plot(hours, prob.T[0], 'm--',
         label='$p(h)$ for zero')
plt.plot(hours, prob.T[1], 'g-.',
         label='$p(h)$ for one')
plt.ylim(-0.2, 1.2)
plt.legend(loc=0);

### Using Logistic Regression to Predict Market Direction

In [ ]:
symbol = 'GLD'

In [ ]:
data = pd.DataFrame(raw[symbol])

In [ ]:
data.rename(columns={symbol: 'price'}, inplace=True)

In [ ]:
data['return'] = np.log(data['price'] / data['price'].shift(1))

In [ ]:
data.dropna(inplace=True)

In [ ]:
lags = 3

In [ ]:
cols = []
for lag in range(1, lags + 1):
    col = 'lag_{}'.format(lag)
    data[col] = data['return'].shift(lag)
    cols.append(col)

In [ ]:
data.dropna(inplace=True)

In [ ]:
from sklearn.metrics import accuracy_score

In [ ]:
lm = linear_model.LogisticRegression(C=1e7, solver='lbfgs',
                                     multi_class='auto',
                                     max_iter=1000)

In [ ]:
lm.fit(data[cols], np.sign(data['return']))

In [ ]:
data['prediction'] = lm.predict(data[cols])

In [ ]:
data['prediction'].value_counts()

In [ ]:
hits = np.sign(data['return'].iloc[lags:] *
               data['prediction'].iloc[lags:]
              ).value_counts()

In [ ]:
hits

In [ ]:
accuracy_score(data['prediction'],
               np.sign(data['return']))

In [ ]:
data['strategy'] = data['prediction'] * data['return']

In [ ]:
data[['return', 'strategy']].sum().apply(np.exp)

In [ ]:
data[['return', 'strategy']].cumsum().apply(np.exp).plot(
                                        figsize=(10, 6));

In [ ]:
data = pd.DataFrame(raw[symbol])

In [ ]:
data.rename(columns={symbol: 'price'}, inplace=True)

In [ ]:
data['return'] = np.log(data['price'] / data['price'].shift(1))

In [ ]:
lags = 5

In [ ]:
cols = []
for lag in range(1, lags + 1):
    col = 'lag_%d' % lag
    data[col] = data['price'].shift(lag)
    cols.append(col)

In [ ]:
data.dropna(inplace=True)

In [ ]:
lm.fit(data[cols], np.sign(data['return']))

In [ ]:
data['prediction'] = lm.predict(data[cols])

In [ ]:
data['prediction'].value_counts()

In [ ]:
hits = np.sign(data['return'].iloc[lags:] *
               data['prediction'].iloc[lags:]
              ).value_counts()

In [ ]:
hits

In [ ]:
accuracy_score(data['prediction'],
               np.sign(data['return']))

In [ ]:
data['strategy'] = data['prediction'] * data['return']

In [ ]:
data[['return', 'strategy']].sum().apply(np.exp)

In [ ]:
data[['return', 'strategy']].cumsum().apply(np.exp).plot(
                                        figsize=(10, 6));

### Generalizing the Approach

In [ ]:
import ScikitVectorBacktester as SCI

In [ ]:
scibt = SCI.ScikitVectorBacktester('EUR=',
                                   '2010-1-1', '2019-12-31',
                                   10000, 0.0, 'logistic')

In [ ]:
scibt.run_strategy('2015-1-1', '2019-12-31',
                   '2015-1-1', '2019-12-31', lags=15)

In [ ]:
scibt.run_strategy('2016-1-1', '2018-12-31',
                   '2019-1-1', '2019-12-31', lags=15)

In [ ]:
scibt.plot_results()

In [ ]:
scibt = SCI.ScikitVectorBacktester('GDX',
                                   '2010-1-1', '2019-12-31',
                                   10000, 0.00, 'logistic')

In [ ]:
scibt.run_strategy('2013-1-1', '2017-12-31',
                   '2018-1-1', '2018-12-31', lags=10)

In [ ]:
scibt.plot_results()

In [ ]:
scibt = SCI.ScikitVectorBacktester('GDX',
                                   '2010-1-1', '2019-12-31',
                                   10000, 0.0025, 'logistic')

In [ ]:
scibt.run_strategy('2013-1-1', '2017-12-31',
                   '2018-1-1', '2018-12-31', lags=10)

In [ ]:
scibt.plot_results()

## Using Deep Learning for Market Movement Prediction

#### The Simple Classification Problem Revisited 

In [ ]:
hours = np.array([0.5, 0.75, 1., 1.25, 1.5, 1.75, 1.75, 2.,
                  2.25, 2.5, 2.75, 3., 3.25, 3.5, 4., 4.25,
                  4.5, 4.75, 5., 5.5])

In [ ]:
success = np.array([0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 1, 0, 1,
                    0, 1, 1, 1, 1, 1, 1]) 

In [ ]:
data = pd.DataFrame({'hours': hours, 'success': success})

In [ ]:
data.info()

In [ ]:
from sklearn.neural_network import MLPClassifier

In [ ]:
model = MLPClassifier(hidden_layer_sizes=[32],
                     max_iter=1000, random_state=100,
                     shuffle=False)

In [ ]:
model.fit(data['hours'].values.reshape(-1, 1), data['success'])

In [ ]:
data['prediction'] = model.predict(data['hours'].values.reshape(-1, 1))

In [ ]:
data.tail()

In [ ]:
data.plot(x='hours', y=['success', 'prediction'],
          style=['ro', 'b-'], ylim=[-.1, 1.1],
          figsize=(10, 6));

### Using Deep Neural Networks to Predict Market Direction

In [ ]:
symbol = 'EUR='

In [ ]:
data = pd.DataFrame(raw[symbol])

In [ ]:
data.rename(columns={symbol: 'price'}, inplace=True)

In [ ]:
data['return'] = np.log(data['price'] /
                         data['price'].shift(1)) 

In [ ]:
data['direction'] = np.where(data['return'] > 0, 1, 0)

In [ ]:
lags = 5

In [ ]:
cols = []
for lag in range(1, lags + 1):
    col = f'lag_{lag}'
    data[col] = data['return'].shift(lag)
    cols.append(col)
data.dropna(inplace=True)

In [ ]:
data.round(4).tail()

In [ ]:
import tensorflow as tf
from tensorflow import keras
from keras.models import Sequential
from keras.layers import Dense
#impo keras.optimizers import adam

In [ ]:
optimizer = keras.optimizers.Adam(learning_rate=0.0001)

In [ ]:
def set_seeds(seed=100):
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(100)

In [ ]:
set_seeds()
model = Sequential()
model.add(Dense(64, activation='relu',
        input_shape=(lags,)))
model.add(Dense(64, activation='relu'))
model.add(Dense(1, activation='sigmoid')) # <5>
model.compile(optimizer=optimizer,
              loss='binary_crossentropy',
              metrics=['accuracy'])

In [ ]:
cutoff = '2017-12-31'

In [ ]:
training_data = data[data.index < cutoff].copy()

In [ ]:
mu, std = training_data.mean(), training_data.std()

In [ ]:
training_data_ = (training_data - mu) / std

In [ ]:
test_data = data[data.index >= cutoff].copy()

In [ ]:
test_data_ = (test_data - mu) / std

In [ ]:
%%time
model.fit(training_data[cols],
          training_data['direction'],
          epochs=50, verbose=False,
          validation_split=0.2, shuffle=False)

In [ ]:
res = pd.DataFrame(model.history.history)

In [ ]:
res[['accuracy', 'val_accuracy']].plot(figsize=(10, 6), style='--');

In [ ]:
model.evaluate(training_data_[cols], training_data['direction'])

In [ ]:
pred = np.where(model.predict(training_data_[cols]) > 0.5, 1, 0)

In [ ]:
pred

In [ ]:
pred[:30].flatten()

In [ ]:
training_data['prediction'] = np.where(pred > 0, 1, -1)

In [ ]:
training_data['strategy'] = (training_data['prediction'] *
                            training_data['return'])

In [ ]:
training_data[['return', 'strategy']].sum().apply(np.exp)

In [ ]:
training_data[['return', 'strategy']].cumsum(
                ).apply(np.exp).plot(figsize=(10, 6));

In [ ]:
model.evaluate(test_data_[cols], test_data['direction'])

In [ ]:
pred = np.where(model.predict(test_data_[cols]) > 0.5, 1, 0)

In [ ]:
test_data['prediction'] = np.where(pred > 0, 1, -1)

In [ ]:
test_data['prediction'].value_counts()

In [ ]:
test_data['strategy'] = (test_data['prediction'] *
                        test_data['return'])

In [ ]:
test_data[['return', 'strategy']].sum().apply(np.exp)

In [ ]:
test_data[['return', 'strategy']].cumsum(
                ).apply(np.exp).plot(figsize=(10, 6));

### Adding Different Types of Features

In [ ]:
data['momentum'] = data['return'].rolling(5).mean().shift(1)

In [ ]:
data['volatility'] = data['return'].rolling(20).std().shift(1)

In [ ]:
data['distance'] = (data['price'] - data['price'].rolling(50).mean()).shift(1)

In [ ]:
# research homework needs to be done to add the "right" features ...

In [ ]:
data.dropna(inplace=True)

In [ ]:
cols.extend(['momentum', 'volatility', 'distance'])

In [ ]:
print(data.round(4).tail())

In [ ]:
training_data = data[data.index < cutoff].copy()

In [ ]:
mu, std = training_data.mean(), training_data.std()

In [ ]:
training_data_ = (training_data - mu) / std

In [ ]:
test_data = data[data.index >= cutoff].copy() 

In [ ]:
test_data_ = (test_data - mu) / std

In [ ]:
set_seeds()
model = Sequential()
model.add(Dense(32, activation='relu',
                input_shape=(len(cols),)))
model.add(Dense(32, activation='relu'))
model.add(Dense(1, activation='sigmoid'))
model.compile(optimizer=optimizer,
              loss='binary_crossentropy',
              metrics=['accuracy'])

In [ ]:
%%time 
model.fit(training_data_[cols], training_data['direction'],
          verbose=False, epochs=25)

In [ ]:
model.evaluate(training_data_[cols], training_data['direction'])

In [ ]:
pred = np.where(model.predict(training_data_[cols]) > 0.5, 1, 0)

In [ ]:
training_data['prediction'] = np.where(pred > 0, 1, -1)

In [ ]:
training_data['strategy'] = training_data['prediction'] * \
                            training_data['return']

In [ ]:
training_data[['return', 'strategy']].sum().apply(np.exp)

In [ ]:
training_data[['return', 'strategy']].cumsum(
                ).apply(np.exp).plot(figsize=(10, 6));

In [ ]:
model.evaluate(test_data_[cols], test_data['direction'])

In [ ]:
pred = np.where(model.predict(test_data_[cols]) > 0.5, 1, 0)

In [ ]:
test_data['prediction'] = np.where(pred > 0, 1, -1)

In [ ]:
test_data['prediction'].value_counts()

In [ ]:
test_data['strategy'] = (test_data['prediction'] *
                        test_data['return'])

In [ ]:
test_data[['return', 'strategy']].sum().apply(np.exp)

In [ ]:
test_data[['return', 'strategy']].cumsum(
                ).apply(np.exp).plot(figsize=(10, 6));

<img src="http://hilpisch.com/tpq_logo.png" alt="The Python Quants" width="35%" align="right" border="0"><br>

<a href="http://tpq.io" target="_blank">http://tpq.io</a> | <a href="http://twitter.com/dyjh" target="_blank">@dyjh</a> | <a href="mailto:training@tpq.io">training@tpq.io</a>